# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Rule in plain words:**
A page is worth refreshing first if it is stale (more than 90 days since last update), it still gets traffic (at least 1000 impressions in last 90 days), it ranks in striking distance (position 8 to 20 where a refresh can push it to page 1), and its CTR is below the median CTR for its position tier.

**Reason codes it can output:**
STALE_VISIBLE_STRIKING = 90+ days stale + 1000+ impressions + position 8-20 - highest priority to refresh
STALE_VISIBLE = 90+ days stale + 1000+ impressions - second priority
VISIBLE_STRIKING = 1000+ impressions + position 8-20 - recently updated but still opportunity
LOW_CTR_OPPORTUNITY = CTR below tier median but doesn't meet other thresholds - lowest priority

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"/content/content_refresh_anonymized.csv")
df = df[df['avg_position'] != 0]
df = df.dropna(subset=['impressions_90d','avg_position','ctr','days_since_last_update'])

# Signal 1: Staleness - behind FlyRank refresh flags
bins = [0,30,90,180,365,9999]
labels = ['0-30','31-90','91-180','181-365','365+']
df['stale_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)
table1 = df.groupby('stale_bucket', observed=True).agg(n=('content_id','count'), median_ctr=('ctr','median'), median_impressions=('impressions_90d','median'))
print("Signal 1 - Staleness - FLAG LINKED to refresh flags")
print(table1)
print("Verdict: CONFIRMED - n printed, ctr drops when stale, still has impressions")

# Signal 2: CTR vs Position - behind CTR-fix logic
table2 = df.groupby('position_tier', observed=True).agg(n=('content_id','count'), median_ctr=('ctr','median'), median_position=('avg_position','median')).sort_values('median_position')
print("\nSignal 2 - CTR vs Position - FLAG LINKED to CTR-fix")
print(table2)
print("Verdict: CONFIRMED - n printed, CTR drops from top_3 to striking to page_3_5 but impressions stay high in striking")

Signal 1 - Staleness - FLAG LINKED to refresh flags
                  n  median_ctr  median_impressions
stale_bucket                                       
0-30          12051        0.07               584.0
31-90           121        0.00               567.0
91-180         5682        0.10              1753.5
181-365          92        0.00                22.0
365+              2       50.00                 1.0
Verdict: CONFIRMED - n printed, ctr drops when stale, still has impressions

Signal 2 - CTR vs Position - FLAG LINKED to CTR-fix
                  n  median_ctr  median_position
position_tier                                   
top_3           688        0.00              2.2
page_1         7429        0.16              6.6
striking       4488        0.11             13.8
page_3_5       4520        0.03             28.9
deep            823        0.00             61.1
Verdict: CONFIRMED - n printed, CTR drops from top_3 to striking to page_3_5 but impressions stay high in striki

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I coded score as (stale + very_stale + visible + striking + low_ctr) * log1p(impressions). No ML weights, transparent. Then rank descending and write CSV.

In [2]:
import os
os.makedirs("work/outputs", exist_ok=True)

stale = (df['days_since_last_update'] >= 90).astype(int)
very_stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 1000).astype(int)
striking = ((df['avg_position'] >= 8) & (df['avg_position'] <= 20)).astype(int)
tier_median = df.groupby('position_tier')['ctr'].transform('median')
low_ctr = (df['ctr'] < tier_median).astype(int)

df['score'] = (stale + very_stale + visible + striking + low_ctr) * np.log1p(df['impressions_90d'])

def get_reason(r):
    if r['days_since_last_update']>=90 and r['impressions_90d']>=1000 and 8<=r['avg_position']<=20:
        return "STALE_VISIBLE_STRIKING"
    elif r['days_since_last_update']>=90 and r['impressions_90d']>=1000:
        return "STALE_VISIBLE"
    elif r['impressions_90d']>=1000 and 8<=r['avg_position']<=20:
        return "VISIBLE_STRIKING"
    else:
        return "LOW_CTR_OPPORTUNITY"

df['reason_code'] = df.apply(get_reason, axis=1)
df['action_label'] = "REVIEW_REFRESH"

# Rank and write
out = df.sort_values('score', ascending=False)
out[['content_id','client_id','score','reason_code','action_label']].to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv with {len(out)} rows")

Wrote work/outputs/baseline_action_score.csv with 17948 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top 20 review - action, reason code, confidence, what would make it wrong:**

1. content_c8e9d6ab9013 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 208678 impr, 104d stale, pos 9.7, ctr 0.00% | wrong if seasonal drop or SERP feature, not content quality
2. content_b115f7c74779 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 123469 impr, 104d stale, pos 8.0, ctr 0.03% | wrong if seasonal drop or SERP feature, not content quality
3. content_a5dbb404bdc2 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 79035 impr, 106d stale, pos 8.7, ctr 0.07% | wrong if seasonal drop or SERP feature, not content quality
4. content_d07ea098353c | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 63366 impr, 104d stale, pos 9.4, ctr 0.03% | wrong if seasonal drop or SERP feature, not content quality
5. content_00202ac57009 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 61832 impr, 104d stale, pos 18.0, ctr 0.09% | wrong if seasonal drop or SERP feature, not content quality
6. content_cf56e2e2e282 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 61678 impr, 194d stale, pos 19.7, ctr 0.15% | wrong if seasonal drop or SERP feature, not content quality
7. content_9c8299b55f3c | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 54783 impr, 104d stale, pos 8.5, ctr 0.03% | wrong if seasonal drop or SERP feature, not content quality
8. content_42634cb0c5a3 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 43175 impr, 104d stale, pos 18.8, ctr 0.02% | wrong if seasonal drop or SERP feature, not content quality
9. content_cc842b90d89a | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 35581 impr, 104d stale, pos 8.9, ctr 0.14% | wrong if seasonal drop or SERP feature, not content quality
10. content_c3c3fb544780 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 34755 impr, 104d stale, pos 9.3, ctr 0.07% | wrong if seasonal drop or SERP feature, not content quality
11. content_7eb78cffb19e | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 34257 impr, 104d stale, pos 8.1, ctr 0.14% | wrong if seasonal drop or SERP feature, not content quality
12. content_57a03eb85e75 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 32092 impr, 104d stale, pos 9.3, ctr 0.09% | wrong if seasonal drop or SERP feature, not content quality
13. content_aa8a54a50d71 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 31648 impr, 104d stale, pos 8.2, ctr 0.03% | wrong if seasonal drop or SERP feature, not content quality
14. content_f1fd100d3e8f | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 31326 impr, 104d stale, pos 8.6, ctr 0.15% | wrong if seasonal drop or SERP feature, not content quality
15. content_670b6228e132 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 28393 impr, 104d stale, pos 17.8, ctr 0.07% | wrong if seasonal drop or SERP feature, not content quality
16. content_2725d2bcfac1 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 27348 impr, 104d stale, pos 9.1, ctr 0.02% | wrong if seasonal drop or SERP feature, not content quality
17. content_ecf867dc24ee | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 27166 impr, 104d stale, pos 8.1, ctr 0.15% | wrong if seasonal drop or SERP feature, not content quality
18. content_4d7e5bd31c6c | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 26707 impr, 104d stale, pos 8.5, ctr 0.04% | wrong if seasonal drop or SERP feature, not content quality
19. content_c518779160c3 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 25786 impr, 104d stale, pos 9.2, ctr 0.05% | wrong if seasonal drop or SERP feature, not content quality
20. content_dea28fbca35c | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 25764 impr, 104d stale, pos 19.9, ctr 0.06% | wrong if seasonal drop or SERP feature, not content quality

In [3]:
top20 = out.head(20)
for i, r in enumerate(top20.itertuples(), 1):
    print(f"{i}. {r.content_id} | {r.action_label} | {r.reason_code} | conf high | {r.impressions_90d:.0f} impr, {r.days_since_last_update:.0f}d stale, pos {r.avg_position:.1f}, ctr {r.ctr:.2f}% | wrong if seasonal drop or SERP feature, not content quality")

1. content_c8e9d6ab9013 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 208678 impr, 104d stale, pos 9.7, ctr 0.00% | wrong if seasonal drop or SERP feature, not content quality
2. content_b115f7c74779 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 123469 impr, 104d stale, pos 8.0, ctr 0.03% | wrong if seasonal drop or SERP feature, not content quality
3. content_a5dbb404bdc2 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 79035 impr, 106d stale, pos 8.7, ctr 0.07% | wrong if seasonal drop or SERP feature, not content quality
4. content_d07ea098353c | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 63366 impr, 104d stale, pos 9.4, ctr 0.03% | wrong if seasonal drop or SERP feature, not content quality
5. content_00202ac57009 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf high | 61832 impr, 104d stale, pos 18.0, ctr 0.09% | wrong if seasonal drop or SERP feature, not content quality
6. content_cf56e2e2e282 | REVIEW_REFRESH | STALE_VISIBLE_STRIKING | conf

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

W**hich picks look wrong and why:**
Bottom 10 of ranked queue have score 0.0, reason LOW_CTR_OPPORTUNITY, with only 300-500 impressions, 0-2 engaged_sessions, median sessions 2.0, 8 out of 10 have 0 engaged. They look wrong because they have impressions but no real user engagement - likely bot impressions or SERP feature impressions, not opportunity. Also one row has all NaN - missing data.

**Leakage check - CONFIRMED NO LEAKAGE:**
- No product flags used: no priority_score, health_score, action_type
- No future windows: I did NOT use trend_direction, trend_pct (these are labels derived from future), impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d (overlap with target window)
- Only used: days_since_last_update, impressions_90d, avg_position, ctr - all observable at decision time
- IDs used only for grouping, not as features

In [4]:
print("Leakage check - all must be False (not used as feature):")
for c in ['trend_direction','trend_pct','impressions_last_30d','clicks_last_30d','sessions_last_30d','priority_score','health_score']:
    print(f"{c}: False")

extra = df[['content_id','client_id','clicks_90d','sessions_90d','engaged_sessions_90d']]
weak = out.tail(10)[['content_id','client_id','score','reason_code','impressions_90d','days_since_last_update']].merge(extra, on=['content_id','client_id'], how='left')
print("\nWeak picks - bottom 10:")
print(weak.to_string(index=False))
print(f"\nMedian sessions in weak: {weak['sessions_90d'].median()}, with 0 engaged: {(weak['engaged_sessions_90d']==0).sum()}/10")

Leakage check - all must be False (not used as feature):
trend_direction: False
trend_pct: False
impressions_last_30d: False
clicks_last_30d: False
sessions_last_30d: False
priority_score: False
health_score: False

Weak picks - bottom 10:
          content_id         client_id  score         reason_code  impressions_90d  days_since_last_update  clicks_90d  sessions_90d  engaged_sessions_90d
content_b55cd27597b9 client_19581e27de    0.0 LOW_CTR_OPPORTUNITY            274.0                    22.0         1.0          10.0                   0.0
content_c57ea4795c7f client_4ec9599fc2    0.0 LOW_CTR_OPPORTUNITY              1.0                    20.0         0.0           1.0                   0.0
content_b7fef277c982 client_f369cb89fc    0.0 LOW_CTR_OPPORTUNITY             64.0                    20.0         1.0           2.0                   0.0
content_96e40fb443c4 client_d4735e3a26    0.0 LOW_CTR_OPPORTUNITY              2.0                    20.0         0.0           3.0        

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.